# BigAlpha2026 H09 submission

Official main-only frozen H09 range-residual implementation.

In [ ]:
def main(datasources, start_date, end_date):
    if not isinstance(datasources, dict):
        raise ValueError("datasources must be a dictionary")
    minute_table = datasources.get("bar1m")
    if not isinstance(minute_table, str) or not minute_table.strip():
        raise ValueError("datasources['bar1m'] is required")

    import numpy as np
    import pandas as pd
    import dai

    OUTPUT_COLUMNS = ["date", "instrument", "factor"]
    factorlib_table = "bigalpha_2026_factorlib"
    universe_table = "bigalpha_2026_instruments"
    epsilon = 1e-12

    def date_bounds(first_date, last_date):
        start = pd.Timestamp(first_date)
        end = pd.Timestamp(last_date)
        if pd.isna(start) or pd.isna(end) or start > end:
            raise ValueError("invalid date range")
        if end == end.normalize():
            end = end + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
        return start, end

    def month_chunks(start, end):
        cursor = start
        while not cursor > end:
            month_end = cursor.to_period("M").end_time
            chunk_end = min(end, month_end)
            yield cursor, chunk_end
            cursor = chunk_end + pd.Timedelta(nanoseconds=1)

    def query(sql, first_date, last_date):
        return dai.query(
            sql,
            filters={"date": [first_date, last_date]},
            compression=True,
        ).df()

    def rank_z(values):
        numeric = pd.to_numeric(values, errors="coerce")
        ranks = numeric.rank(method="average")
        centered = ranks - ranks.mean()
        std = float(centered.std(ddof=0)) if centered.notna().any() else np.nan
        if not np.isfinite(std) or not std > epsilon:
            return pd.Series(np.nan, index=values.index, dtype="float64")
        return centered / std

    def compute_chunk(minute_table, start, end):
        minute_start = start.strftime("%Y-%m-%d %H:%M:%S")
        minute_end = end.strftime("%Y-%m-%d %H:%M:%S.%f")
        day_start = start.date().isoformat()
        day_end = end.date().isoformat()
        minute = query(
            f"SELECT date, instrument, high, low, pre_close FROM {minute_table} "
            f"WHERE date >= '{minute_start}' AND NOT date > '{minute_end}' "
            "ORDER BY date, instrument",
            minute_start,
            minute_end,
        )
        controls = query(
            f"SELECT date, instrument, volatility_5 FROM {factorlib_table} "
            f"WHERE date >= '{day_start}' AND NOT date > '{day_end}' "
            "ORDER BY date, instrument",
            day_start,
            day_end,
        )
        universe = query(
            f"SELECT date, instrument FROM {universe_table} "
            f"WHERE date >= '{day_start}' AND NOT date > '{day_end}' "
            "ORDER BY date, instrument",
            day_start,
            day_end,
        )
        required = {"date", "instrument", "high", "low", "pre_close"}
        if not required.issubset(minute.columns):
            raise ValueError("official minute query lacks required columns")
        if not {"date", "instrument", "volatility_5"}.issubset(controls.columns):
            raise ValueError("official factorlib query lacks required columns")
        if not {"date", "instrument"}.issubset(universe.columns):
            raise ValueError("official universe query lacks required columns")
        work = minute.loc[:, ["date", "instrument", "high", "low", "pre_close"]].copy()
        public = controls.loc[:, ["date", "instrument", "volatility_5"]].copy()
        membership = universe.loc[:, ["date", "instrument"]].copy()
        for frame in (work, public, membership):
            frame["date"] = pd.to_datetime(frame["date"], errors="coerce").dt.normalize()
            if frame["date"].isna().any() or frame["instrument"].isna().any():
                raise ValueError("official query contains invalid keys")
            frame["instrument"] = frame["instrument"].astype(str)
        if membership.duplicated(["date", "instrument"]).any():
            raise ValueError("official universe contains duplicate membership keys")
        if public.duplicated(["date", "instrument"]).any():
            raise ValueError("official factorlib contains duplicate keys")
        work = work.merge(
            membership, on=["date", "instrument"], how="inner", validate="many_to_one"
        )
        public = public.merge(
            membership, on=["date", "instrument"], how="inner", validate="one_to_one"
        )
        for column in ("high", "low", "pre_close"):
            numeric = pd.to_numeric(work[column], errors="coerce")
            work[column] = numeric.where(np.isfinite(numeric))
        volatility = pd.to_numeric(public["volatility_5"], errors="coerce")
        public["volatility_5"] = volatility.where(np.isfinite(volatility))
        work["valid_pre_close"] = work["pre_close"].where(work["pre_close"] > 0)
        daily = (
            work.groupby(["date", "instrument"], sort=True)
            .agg(
                high_max=("high", "max"),
                low_min=("low", "min"),
                avg_valid_pre_close=("valid_pre_close", "mean"),
            )
            .reset_index()
        )
        daily["daily_range"] = (
            daily["high_max"] - daily["low_min"]
        ) / daily["avg_valid_pre_close"]
        merged = daily.loc[:, ["date", "instrument", "daily_range"]].merge(
            public, on=["date", "instrument"], how="inner", validate="one_to_one"
        )
        pieces = []
        for _, day in merged.groupby("date", sort=True):
            current = day.loc[:, ["date", "instrument", "daily_range", "volatility_5"]].copy()
            current["factor"] = np.nan
            finite = np.isfinite(current["daily_range"]) & np.isfinite(current["volatility_5"])
            usable = current.loc[finite].copy()
            if len(usable) >= 2:
                y = rank_z(usable["daily_range"])
                x = rank_z(usable["volatility_5"])
                if y.notna().all() and x.notna().all():
                    x_values = x.to_numpy(dtype=float)
                    y_values = y.to_numpy(dtype=float)
                    denominator = float(np.dot(x_values, x_values))
                    if denominator > epsilon:
                        beta = float(np.dot(x_values, y_values) / denominator)
                        alpha = float(y.mean() - beta * x.mean())
                        # Frozen direction is negative; the platform expects higher-is-better.
                        current.loc[usable.index, "factor"] = -(y - (alpha + beta * x))
            pieces.append(current.loc[:, OUTPUT_COLUMNS])
        output = (
            pd.concat(pieces, ignore_index=True)
            if pieces
            else pd.DataFrame(columns=OUTPUT_COLUMNS)
        )
        output["factor"] = pd.to_numeric(output["factor"], errors="coerce")
        output = output.loc[np.isfinite(output["factor"]), OUTPUT_COLUMNS].copy()
        if output.duplicated(["date", "instrument"]).any():
            raise ValueError("factor output has duplicate keys")
        return output.sort_values(
            ["date", "instrument"], kind="mergesort"
        ).reset_index(drop=True)

    start, end = date_bounds(start_date, end_date)
    pieces = []
    for chunk_start, chunk_end in month_chunks(start, end):
        chunk = compute_chunk(minute_table, chunk_start, chunk_end)
        if not chunk.empty:
            pieces.append(chunk)
    output = (
        pd.concat(pieces, ignore_index=True)
        if pieces
        else pd.DataFrame(columns=OUTPUT_COLUMNS)
    )
    if output.duplicated(["date", "instrument"]).any():
        raise ValueError("factor output has duplicate keys across calendar-month chunks")
    return output.loc[:, OUTPUT_COLUMNS].sort_values(
        ["date", "instrument"], kind="mergesort"
    ).reset_index(drop=True)
